# CrisisCompute — Multi-Agent Compute Negotiation Training

**OpenEnv Hackathon India 2026** | Theme #1 (Multi-Agent Interactions) + Theme #4 (Self-Improvement)

---

## How to Run on Google Colab

1. Open [Google Colab](https://colab.research.google.com)
2. **File > Upload Notebook** > select this `.ipynb` file
3. **Runtime > Change runtime type** > select **T4 GPU**
4. If using LLM or Hybrid mode, paste your `HF_TOKEN` in the config cell below
5. **Runtime > Run all** (Ctrl+F9) — takes ~15-25 min
6. Key outputs for judges will appear in the `results/` folder

---

## Problem

Three ML pipeline agents — **Data Loader**, **Data Cleaner**, and **ML Trainer** — share a single compute cluster with limited **CPU, GPU, and memory**. Each agent has tasks with deadlines and must **negotiate** for resources under pressure. The challenge: maximize team throughput while maintaining **fairness**, handling **crises** (GPU outages, urgent tasks), and building accurate **beliefs** about what other agents need (Theory of Mind).

## Architecture

```
+-------------------------------------------------+
|            SatyaEnv (Compute Cluster)            |
|  Resources: 16 CPU | 32GB RAM | 1 GPU           |
|  Tasks: deadline-sensitive, priority-weighted    |
|  Events: GPU outage, urgent injection, conflict  |
+-------------------------------------------------+
|  data_loader <-> data_cleaner <-> ml_trainer     |
|  Negotiation Protocol (propose/accept/reject)    |
|  Reputation + Belief System (Theory of Mind)     |
|  Coalition Formation (resource sharing)          |
+-------------------------------------------------+
```

---

## The Three Agent Modes

### Mode 1: Pure RL (Q-Learning)

Each agent maintains a **Q-table** mapping `(state, action) -> expected reward`. Learning via Bellman:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \big[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \big]$$

Epsilon-greedy exploration with decay (0.35 -> 0.19). Q-tables persist across runs.

### Mode 2: Pure LLM (Language Model Agents)

Each agent is backed by an LLM (Llama-3.1-8B via HuggingFace Inference). The agent receives environment state as a prompt and returns a JSON action. Learning via episodic memory and temperature adaptation (0.7 -> 0.2).

### Mode 3: Hybrid (LLM + RL)

Best of both worlds:
1. **LLM** analyzes state and proposes a strategy hint
2. **RL agent** uses the hint as a bias in exploration
3. Only **RL learns** (updates Q-values) — LLM provides guidance

### Adaptive Curriculum (Theme #4)

Training progresses through 5 difficulty levels (negotiation off -> on, crisis events off -> on). Self-play snapshots compare current vs past policy.

---

## Step 1 — Setup & Dependencies

In [ ]:
!pip install -q numpy matplotlib requests pydantic python-dotenv groq openenv-core>=0.1.13
!pip install -q unsloth trl peft accelerate bitsandbytes datasets
print('\n--- All dependencies installed ---')

In [ ]:
import os, sys, subprocess, json, importlib
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

REPO_URL  = 'https://github.com/Ashrua7-7/CrisisCompute.git'
REPO_PATH = '/content/multi-agent'

if not os.path.exists(REPO_PATH):
    print('Cloning CrisisCompute...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_PATH], check=True)
else:
    print('Repository already cloned')

os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)

from src.evaluate import MetricsCalculator, LearningAnalyzer
from src.visualize import ResultsVisualizer
from PIL import Image

def show_image(path, title=None):
    """Display a saved image inline in Colab."""
    if Path(path).exists():
        fig, ax = plt.subplots(figsize=(14, 8))
        ax.imshow(Image.open(path))
        if title:
            ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f'Image not found: {path}')

print(f'Working directory: {os.getcwd()}')
print(f'Python: {sys.version.split()[0]}')

---

## Step 2 — Configure

In [ ]:
# === API KEY (required for LLM and Hybrid modes) ===
# Paste your HuggingFace token here:
# os.environ['HF_TOKEN'] = 'hf_...'   # from huggingface.co/settings/tokens

# === Experiment Tracking ===
# USE_WANDB = False  → uses TensorBoard (no account needed, logs saved locally)
# USE_WANDB = True   → uses Weights & Biases (requires wandb login below)
USE_WANDB = False
# import wandb; wandb.login(key='YOUR_WANDB_API_KEY')  # wandb.ai/settings

# Common settings
os.environ['LLM_PROVIDER']              = 'huggingface'
os.environ['HUGGINGFACE_MODEL']         = 'meta-llama/Llama-3.1-8B-Instruct'
os.environ['HUGGINGFACE_BASE_URL']      = 'https://router.huggingface.co/v1'
os.environ['NEGOTIATION_ENABLED']       = 'true'
os.environ['SHOW_PHASE_LOGS']           = 'true'
os.environ['CURRICULUM_PHASE_EPISODES'] = '8'

if USE_WANDB:
    os.environ['WANDB_PROJECT'] = 'crisiscompute-openenv'
    print('Tracking: Weights & Biases')
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print('Tracking: TensorBoard (local)')

print('Configuration set.')

---

# PART A — RL Training: BEFORE vs AFTER

We first run **untrained (fresh) agents** to get a baseline, then train them for 100 episodes and compare.

## Step 3A — BEFORE RL: Baseline (Untrained Agents)

Run fresh RL agents with **no self-improvement, no curriculum, fresh Q-tables**. This captures what agents do with ZERO training.

In [ ]:
# --- BEFORE RL: Fresh untrained agents ---
os.environ['TRAINING_AGENT_MODE']       = 'rl'
os.environ['NUM_EPISODES']              = '10'
os.environ['SELF_IMPROVEMENT_ENABLED']  = 'false'

# Delete any saved Q-tables so agents start completely fresh
import glob
for f in glob.glob('q_tables/*.json'):
    os.remove(f)
    print(f'  Deleted {f}')

# Force reload train module with new env vars
if 'train' in sys.modules:
    del sys.modules['train']
import train

print('\n' + '='*70)
print('BEFORE RL: Running 10 episodes with UNTRAINED agents')
print('='*70)

baseline_rl_result = train.train_agents(num_episodes=10)

# Save baseline results
with open('results/training_results.json') as f:
    baseline_rl_episodes = json.load(f)

baseline_rl_rewards     = [ep['total_reward'] for ep in baseline_rl_episodes]
baseline_rl_completions = [MetricsCalculator.calculate_completion_rate(ep) for ep in baseline_rl_episodes]
baseline_rl_ontimes     = [MetricsCalculator.calculate_on_time_rate(ep) for ep in baseline_rl_episodes]
baseline_rl_fairness    = [ep.get('avg_fairness_score', 0) for ep in baseline_rl_episodes]

print(f'\n--- BEFORE RL BASELINE ---')
print(f'  Mean Reward:     {np.mean(baseline_rl_rewards):.1f}')
print(f'  Mean Completion: {np.mean(baseline_rl_completions):.1f}%')
print(f'  Mean On-Time:    {np.mean(baseline_rl_ontimes):.1f}%')
print(f'  Mean Fairness:   {np.mean(baseline_rl_fairness):.3f}')

## Step 3B — AFTER RL: Train for 100 Episodes

Now we train the same RL agents for **100 episodes** with **adaptive curriculum** and **self-play** (Theme #4). Q-tables accumulate knowledge across episodes.

In [ ]:
# --- AFTER RL: Full training ---
os.environ['TRAINING_AGENT_MODE']       = 'rl'
os.environ['NUM_EPISODES']              = '100'
os.environ['SELF_IMPROVEMENT_ENABLED']  = 'true'

# Force reload train module
if 'train' in sys.modules:
    del sys.modules['train']
import train

print('\n' + '='*70)
print('AFTER RL: Training 100 episodes with curriculum + self-play')
print('='*70)

trained_rl_result = train.train_agents(num_episodes=100)

# Save trained results
with open('results/training_results.json') as f:
    trained_rl_episodes = json.load(f)

trained_rl_rewards     = [ep['total_reward'] for ep in trained_rl_episodes]
trained_rl_completions = [MetricsCalculator.calculate_completion_rate(ep) for ep in trained_rl_episodes]
trained_rl_ontimes     = [MetricsCalculator.calculate_on_time_rate(ep) for ep in trained_rl_episodes]
trained_rl_fairness    = [ep.get('avg_fairness_score', 0) for ep in trained_rl_episodes]

# Use last 10 episodes as "after training" metrics
last_n = 10
print(f'\n--- AFTER RL (last {last_n} episodes) ---')
print(f'  Mean Reward:     {np.mean(trained_rl_rewards[-last_n:]):.1f}')
print(f'  Mean Completion: {np.mean(trained_rl_completions[-last_n:]):.1f}%')
print(f'  Mean On-Time:    {np.mean(trained_rl_ontimes[-last_n:]):.1f}%')
print(f'  Mean Fairness:   {np.mean(trained_rl_fairness[-last_n:]):.3f}')

In [ ]:
# Display auto-generated plots from RL training
show_image('results/reward_curve.png', 'RL Training: Auto-Generated Reward Curve')
show_image('results/metrics_dashboard.png', 'RL Training: Metrics Dashboard')

## Step 3C — RL: BEFORE vs AFTER Comparison

Direct visual comparison showing that **RL training improved agent performance**.

In [ ]:
last_n = 10

before_rl = {
    'Reward':        np.mean(baseline_rl_rewards),
    'Completion %':  np.mean(baseline_rl_completions),
    'On-Time %':     np.mean(baseline_rl_ontimes),
    'Fairness':      np.mean(baseline_rl_fairness) * 100,
}
after_rl = {
    'Reward':        np.mean(trained_rl_rewards[-last_n:]),
    'Completion %':  np.mean(trained_rl_completions[-last_n:]),
    'On-Time %':     np.mean(trained_rl_ontimes[-last_n:]),
    'Fairness':      np.mean(trained_rl_fairness[-last_n:]) * 100,
}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('RL MODE: BEFORE Training vs AFTER Training (100 episodes)',
             fontsize=16, fontweight='bold')

for ax, metric in zip(axes, before_rl.keys()):
    b_val = before_rl[metric]
    a_val = after_rl[metric]
    bars = ax.bar(['BEFORE\n(untrained)', 'AFTER\n(100 eps)'],
                  [b_val, a_val],
                  color=['#ff6b6b', '#2ecc71'], width=0.5,
                  edgecolor='white', linewidth=2)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    # Annotate values on bars
    for bar, val in zip(bars, [b_val, a_val]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}', ha='center', fontsize=12, fontweight='bold')

    # Improvement arrow
    delta = a_val - b_val
    pct = (delta / abs(b_val) * 100) if b_val != 0 else 0
    color = 'darkgreen' if delta > 0 else 'red'
    ax.text(0.5, max(b_val, a_val) * 1.08,
            f'{delta:+.1f} ({pct:+.1f}%)',
            ha='center', fontsize=11, fontweight='bold', color=color,
            transform=ax.get_xaxis_transform())

plt.tight_layout()
plt.savefig('results/rl_before_after.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/rl_before_after.png')

In [ ]:
# RL Reward Curve with annotations
fig, ax = plt.subplots(figsize=(14, 5))

ep_nums = list(range(1, len(trained_rl_rewards) + 1))
ax.plot(ep_nums, trained_rl_rewards, alpha=0.3, color='royalblue', label='Per-episode reward')

# Smoothed line
w = 5
if len(trained_rl_rewards) >= w:
    smoothed = np.convolve(trained_rl_rewards, np.ones(w)/w, mode='valid')
    ax.plot(range(w, len(trained_rl_rewards) + 1), smoothed,
            color='royalblue', lw=2.5, label=f'Smoothed (window={w})')

# Annotate ep1 and peak
peak_idx = np.argmax(trained_rl_rewards)
ax.annotate(f'EP1: {trained_rl_rewards[0]:.0f}',
            xy=(1, trained_rl_rewards[0]), fontsize=11, fontweight='bold',
            color='red', ha='left',
            arrowprops=dict(arrowstyle='->', color='red'),
            xytext=(5, trained_rl_rewards[0] - 30))
ax.annotate(f'PEAK EP{peak_idx+1}: {trained_rl_rewards[peak_idx]:.0f}',
            xy=(peak_idx+1, trained_rl_rewards[peak_idx]), fontsize=11, fontweight='bold',
            color='darkgreen', ha='center',
            arrowprops=dict(arrowstyle='->', color='darkgreen'),
            xytext=(peak_idx+1, trained_rl_rewards[peak_idx] + 30))

ax.set_title('RL Training: Reward Curve (100 Episodes)', fontsize=14, fontweight='bold')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Total Reward', fontsize=12)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/rl_reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/rl_reward_curve.png')

---

# PART B — HYBRID Training: BEFORE vs AFTER

Now we repeat the same experiment with **Hybrid mode** (LLM strategy hints + RL learning) to show it performs even better.

## Step 4A — BEFORE HYBRID: Baseline (Untrained Agents)

In [ ]:
# --- BEFORE HYBRID: Fresh untrained agents ---
os.environ['TRAINING_AGENT_MODE']       = 'hybrid'
os.environ['NUM_EPISODES']              = '10'
os.environ['SELF_IMPROVEMENT_ENABLED']  = 'false'

# Delete Q-tables so agents start fresh
for f in glob.glob('q_tables/*.json'):
    os.remove(f)

if 'train' in sys.modules:
    del sys.modules['train']
import train

print('\n' + '='*70)
print('BEFORE HYBRID: Running 10 episodes with UNTRAINED agents')
print('='*70)

baseline_hybrid_result = train.train_agents(num_episodes=10)

with open('results/training_results.json') as f:
    baseline_hybrid_episodes = json.load(f)

baseline_hyb_rewards     = [ep['total_reward'] for ep in baseline_hybrid_episodes]
baseline_hyb_completions = [MetricsCalculator.calculate_completion_rate(ep) for ep in baseline_hybrid_episodes]
baseline_hyb_ontimes     = [MetricsCalculator.calculate_on_time_rate(ep) for ep in baseline_hybrid_episodes]
baseline_hyb_fairness    = [ep.get('avg_fairness_score', 0) for ep in baseline_hybrid_episodes]

print(f'\n--- BEFORE HYBRID BASELINE ---')
print(f'  Mean Reward:     {np.mean(baseline_hyb_rewards):.1f}')
print(f'  Mean Completion: {np.mean(baseline_hyb_completions):.1f}%')
print(f'  Mean Fairness:   {np.mean(baseline_hyb_fairness):.3f}')

## Step 4B — AFTER HYBRID: Train for 100 Episodes

In [ ]:
# --- AFTER HYBRID: Full training ---
os.environ['TRAINING_AGENT_MODE']       = 'hybrid'
os.environ['NUM_EPISODES']              = '100'
os.environ['SELF_IMPROVEMENT_ENABLED']  = 'true'

if 'train' in sys.modules:
    del sys.modules['train']
import train

print('\n' + '='*70)
print('AFTER HYBRID: Training 100 episodes with curriculum + self-play')
print('='*70)

trained_hybrid_result = train.train_agents(num_episodes=100)

with open('results/training_results.json') as f:
    trained_hybrid_episodes = json.load(f)

trained_hyb_rewards     = [ep['total_reward'] for ep in trained_hybrid_episodes]
trained_hyb_completions = [MetricsCalculator.calculate_completion_rate(ep) for ep in trained_hybrid_episodes]
trained_hyb_ontimes     = [MetricsCalculator.calculate_on_time_rate(ep) for ep in trained_hybrid_episodes]
trained_hyb_fairness    = [ep.get('avg_fairness_score', 0) for ep in trained_hybrid_episodes]

last_n = 10
print(f'\n--- AFTER HYBRID (last {last_n} episodes) ---')
print(f'  Mean Reward:     {np.mean(trained_hyb_rewards[-last_n:]):.1f}')
print(f'  Mean Completion: {np.mean(trained_hyb_completions[-last_n:]):.1f}%')
print(f'  Mean Fairness:   {np.mean(trained_hyb_fairness[-last_n:]):.3f}')

In [ ]:
# Display auto-generated plots from Hybrid training
show_image('results/reward_curve.png', 'Hybrid Training: Auto-Generated Reward Curve')
show_image('results/metrics_dashboard.png', 'Hybrid Training: Metrics Dashboard')

## Step 4C — HYBRID: BEFORE vs AFTER Comparison

In [ ]:
last_n = 10

before_hyb = {
    'Reward':        np.mean(baseline_hyb_rewards),
    'Completion %':  np.mean(baseline_hyb_completions),
    'On-Time %':     np.mean(baseline_hyb_ontimes),
    'Fairness':      np.mean(baseline_hyb_fairness) * 100,
}
after_hyb = {
    'Reward':        np.mean(trained_hyb_rewards[-last_n:]),
    'Completion %':  np.mean(trained_hyb_completions[-last_n:]),
    'On-Time %':     np.mean(trained_hyb_ontimes[-last_n:]),
    'Fairness':      np.mean(trained_hyb_fairness[-last_n:]) * 100,
}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('HYBRID MODE: BEFORE Training vs AFTER Training (100 episodes)',
             fontsize=16, fontweight='bold')

for ax, metric in zip(axes, before_hyb.keys()):
    b_val = before_hyb[metric]
    a_val = after_hyb[metric]
    bars = ax.bar(['BEFORE\n(untrained)', 'AFTER\n(100 eps)'],
                  [b_val, a_val],
                  color=['#ff6b6b', '#3498db'], width=0.5,
                  edgecolor='white', linewidth=2)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    for bar, val in zip(bars, [b_val, a_val]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}', ha='center', fontsize=12, fontweight='bold')

    delta = a_val - b_val
    pct = (delta / abs(b_val) * 100) if b_val != 0 else 0
    color = 'darkgreen' if delta > 0 else 'red'
    ax.text(0.5, max(b_val, a_val) * 1.08,
            f'{delta:+.1f} ({pct:+.1f}%)',
            ha='center', fontsize=11, fontweight='bold', color=color,
            transform=ax.get_xaxis_transform())

plt.tight_layout()
plt.savefig('results/hybrid_before_after.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/hybrid_before_after.png')

In [ ]:
# Hybrid Reward Curve with annotations
fig, ax = plt.subplots(figsize=(14, 5))

ep_nums = list(range(1, len(trained_hyb_rewards) + 1))
ax.plot(ep_nums, trained_hyb_rewards, alpha=0.3, color='#3498db', label='Per-episode reward')

w = 5
if len(trained_hyb_rewards) >= w:
    smoothed = np.convolve(trained_hyb_rewards, np.ones(w)/w, mode='valid')
    ax.plot(range(w, len(trained_hyb_rewards) + 1), smoothed,
            color='#3498db', lw=2.5, label=f'Smoothed (window={w})')

peak_idx = np.argmax(trained_hyb_rewards)
ax.annotate(f'EP1: {trained_hyb_rewards[0]:.0f}',
            xy=(1, trained_hyb_rewards[0]), fontsize=11, fontweight='bold',
            color='red', ha='left',
            arrowprops=dict(arrowstyle='->', color='red'),
            xytext=(5, trained_hyb_rewards[0] - 30))
ax.annotate(f'PEAK EP{peak_idx+1}: {trained_hyb_rewards[peak_idx]:.0f}',
            xy=(peak_idx+1, trained_hyb_rewards[peak_idx]), fontsize=11, fontweight='bold',
            color='darkgreen', ha='center',
            arrowprops=dict(arrowstyle='->', color='darkgreen'),
            xytext=(peak_idx+1, trained_hyb_rewards[peak_idx] + 30))

ax.set_title('Hybrid Training: Reward Curve (100 Episodes)', fontsize=14, fontweight='bold')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Total Reward', fontsize=12)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/hybrid_reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/hybrid_reward_curve.png')

---

# PART C — RL vs HYBRID: Head-to-Head Comparison

Which mode learns better? Comparing trained RL vs trained Hybrid side-by-side.

In [ ]:
last_n = 10

rl_final = {
    'Reward':        np.mean(trained_rl_rewards[-last_n:]),
    'Completion %':  np.mean(trained_rl_completions[-last_n:]),
    'On-Time %':     np.mean(trained_rl_ontimes[-last_n:]),
    'Fairness':      np.mean(trained_rl_fairness[-last_n:]) * 100,
}
hyb_final = {
    'Reward':        np.mean(trained_hyb_rewards[-last_n:]),
    'Completion %':  np.mean(trained_hyb_completions[-last_n:]),
    'On-Time %':     np.mean(trained_hyb_ontimes[-last_n:]),
    'Fairness':      np.mean(trained_hyb_fairness[-last_n:]) * 100,
}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('HEAD-TO-HEAD: RL vs HYBRID (last 10 episodes after 100-ep training)',
             fontsize=16, fontweight='bold')

for ax, metric in zip(axes, rl_final.keys()):
    r_val = rl_final[metric]
    h_val = hyb_final[metric]
    bars = ax.bar(['RL\n(Q-learning)', 'HYBRID\n(LLM + RL)'],
                  [r_val, h_val],
                  color=['#2ecc71', '#3498db'], width=0.5,
                  edgecolor='white', linewidth=2)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    for bar, val in zip(bars, [r_val, h_val]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}', ha='center', fontsize=12, fontweight='bold')

    winner = 'HYBRID' if h_val > r_val else 'RL'
    delta = abs(h_val - r_val)
    ax.text(0.5, max(r_val, h_val) * 1.08,
            f'{winner} wins (+{delta:.1f})',
            ha='center', fontsize=10, fontweight='bold', color='purple',
            transform=ax.get_xaxis_transform())

plt.tight_layout()
plt.savefig('results/rl_vs_hybrid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/rl_vs_hybrid.png')

In [ ]:
# Overlaid reward curves: RL vs Hybrid
fig, ax = plt.subplots(figsize=(14, 5))

w = 5
# RL
ax.plot(range(1, len(trained_rl_rewards)+1), trained_rl_rewards,
        alpha=0.15, color='#2ecc71')
if len(trained_rl_rewards) >= w:
    s_rl = np.convolve(trained_rl_rewards, np.ones(w)/w, mode='valid')
    ax.plot(range(w, len(trained_rl_rewards)+1), s_rl,
            color='#2ecc71', lw=2.5, label='RL (smoothed)')

# Hybrid
ax.plot(range(1, len(trained_hyb_rewards)+1), trained_hyb_rewards,
        alpha=0.15, color='#3498db')
if len(trained_hyb_rewards) >= w:
    s_hyb = np.convolve(trained_hyb_rewards, np.ones(w)/w, mode='valid')
    ax.plot(range(w, len(trained_hyb_rewards)+1), s_hyb,
            color='#3498db', lw=2.5, label='Hybrid (smoothed)')

ax.set_title('Reward Curves: RL vs Hybrid Training', fontsize=14, fontweight='bold')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Total Reward', fontsize=12)
ax.legend(fontsize=12)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/rl_vs_hybrid_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/rl_vs_hybrid_curves.png')

---
## Step 5 — 3-Mode Comparison: LLM vs RL vs Hybrid

Headline plot for judges: runs all three agent modes on the **same seed and curriculum**,
overlays smoothed reward curves, and reports mean reward / completion rate / stable-phase
improvement per mode. This is the single artifact that proves all three modes work and
quantifies which performs best.

> Runs `MODE_COMPARE=1` with 20 episodes per mode (~3–5 min total). LLM/Hybrid require
> a valid `HF_TOKEN` or `GROQ_API_KEY` from Step 2 to call the LLM provider.

In [ ]:
# 3-Mode Comparison: LLM vs RL vs Hybrid
# ------------------------------------------------------------------
# Runs all three modes on the same seed/curriculum and produces a
# unified comparison plot at results/mode_comparison.png
import os, sys, subprocess, shutil, json
from pathlib import Path

os.environ['MODE_COMPARE']           = '1'
os.environ['MODE_COMPARE_EPISODES']  = '20'   # per mode (~3-5 min total)
os.environ['MODE_COMPARE_SEED']      = '42'
os.environ['RL_WARMUP_EPISODES']     = '15'   # quick warmup so curves don't peak at EP1

if Path('q_tables').exists():
    shutil.rmtree('q_tables')
if Path('results/mode_comparison.png').exists():
    Path('results/mode_comparison.png').unlink()

print('Starting 3-mode comparison run (LLM, RL, Hybrid)...')
subprocess.run([sys.executable, 'train.py'], check=False)

show_image('results/mode_comparison.png',
           '3-Mode Comparison: LLM vs RL vs Hybrid (same seed)')

summary_path = Path('results/mode_compare_summary.json')
if summary_path.exists():
    cmp_data = json.loads(summary_path.read_text())
    print(f"\nSeed: {cmp_data['seed']}  |  Episodes per mode: {cmp_data['episodes']}\n")
    print(f"{'Mode':<8} {'mean_reward':>12} {'mean_completion':>16} {'improvement':>13}")
    print('-' * 52)
    for m in cmp_data['modes']:
        print(f"{m['mode'].upper():<8} "
              f"{m['mean_reward']:>12.1f} "
              f"{m['mean_completion']:>15.1f}% "
              f"{m['reward_improvement_pct']:>+12.1f}%")
    best = max(cmp_data['modes'], key=lambda m: m['mean_reward'])
    print(f"\nBest mean reward: {best['mode'].upper()} @ {best['mean_reward']:.1f}")

---

# PART D — Theme #4: Self-Improvement Analysis

In [ ]:
theme4_path = Path('results/theme4_summary.json')
if theme4_path.exists():
    with open(theme4_path) as f:
        theme4 = json.load(f)

    print('THEME #4: SELF-IMPROVEMENT')
    print('='*70)

    print('\nCurriculum Progression:')
    for entry in theme4.get('curriculum_history', []):
        status = 'Promoted' if entry['promoted'] else 'Held'
        print(f"  Phase {entry['phase']:2d} -> Level {entry['level']} | "
              f"Completion: {entry['completion']:.1f}% | "
              f"Fairness: {entry.get('fairness', 0):.2f} | {status}")

    print('\nHoldout Improvement (Trained vs Fresh):')
    deltas = theme4.get('holdout_delta', {})
    for key in ['avg_total_reward', 'avg_completion_rate', 'avg_on_time_rate', 'avg_fairness_score']:
        label = key.replace('avg_', '').replace('_', ' ').title()
        print(f'  {label:25s}: {deltas.get(key, 0):+.2f}')
    print('='*70)
else:
    print('Theme #4 summary not found (only generated in RL/Hybrid mode with self-improvement)')

---

# PART E — Unsloth Fine-Tuning (SFT + GRPO)

Fine-tune a small LLM on the generated trajectories:
- **SFT**: Supervised fine-tuning on top-50% reward episodes
- **GRPO**: Group Relative Policy Optimization — directly optimizes environment reward

> Requires Colab **T4 GPU** (free tier works for 1B models)

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME   = 'unsloth/Llama-3.2-1B-Instruct'
MAX_SEQ_LEN  = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Loaded {MODEL_NAME}')
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# Use the hybrid training episodes (most recent run) for fine-tuning data
from datasets import Dataset

episodes_for_ft = trained_hybrid_episodes
rewards_for_ft  = trained_hyb_rewards

SYSTEM_PROMPT = """You are a compute resource allocation agent in a multi-agent ML pipeline.
A shared cluster has limited GPU, CPU, and memory shared among three agents.
Your goal: complete your tasks on time while negotiating fairly with other agents.

Available actions:
- request_resources: claim CPU cores, memory, and optionally GPU
- wait: hold off this hour
- yield_to_urgent: give up resources for an urgent task

Respond ONLY with a valid JSON action object."""

AGENT_ROLES = {
    'data_loader':  'You ingest raw datasets. LOW resource needs (2 CPU, 4 GB RAM, no GPU).',
    'data_cleaner': 'You pre-process datasets. MEDIUM resource needs (4 CPU, 8 GB RAM, no GPU).',
    'ml_trainer':   'You train ML models. HIGH resource needs (2 CPU, 16 GB RAM, 1 GPU).',
}

r_min, r_max = min(rewards_for_ft), max(rewards_for_ft)
def norm_reward(r):
    return float(np.clip((r - r_min) / (r_max - r_min), 0, 1)) if r_max > r_min else 0.5

median_r = float(np.median(rewards_for_ft))
sft_samples, grpo_samples = [], []

for ep in episodes_for_ft:
    ep_r = ep['total_reward']
    nr   = norm_reward(ep_r)
    for step in ep.get('steps', []):
        hour    = step.get('hour', 1)
        actions = step.get('actions', {})
        for agent_id, action in actions.items():
            if action is None:
                continue
            role = AGENT_ROLES.get(agent_id, 'Resource allocation agent.')
            user_msg = (f"Hour {hour}/8 | Scenario: {ep.get('scenario', 'stable')} | "
                        f"Level: {ep.get('curriculum_level', 0)}\n"
                        f"Your role: {role}\nWhat action do you take?")
            msgs = [{'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': user_msg}]
            action_str = json.dumps(action) if isinstance(action, dict) else str(action)
            grpo_samples.append({'prompt': msgs, 'completion': action_str, 'reward': nr})
            if ep_r >= median_r:
                sft_samples.append({'prompt': msgs, 'completion': action_str})

print(f'GRPO samples: {len(grpo_samples)}')
print(f'SFT samples (top 50%): {len(sft_samples)}')

In [ ]:
# SFT Training
from trl import SFTConfig, SFTTrainer

def format_sft(sample):
    msgs = sample['prompt'] + [{'role': 'assistant', 'content': sample['completion']}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

sft_ds = Dataset.from_list(sft_samples).shuffle(seed=42)
sft_ds = sft_ds.map(lambda x: {'text': format_sft(x)})
sft_split = sft_ds.train_test_split(test_size=0.1, seed=42)

print(f'SFT train: {len(sft_split["train"])} | eval: {len(sft_split["test"])}')

sft_tracker = 'wandb' if USE_WANDB else 'tensorboard'

sft_trainer = SFTTrainer(
    model           = model,
    args            = SFTConfig(
        output_dir                  = 'crisiscompute_sft',
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 1,
        learning_rate               = 2e-4,
        warmup_ratio                = 0.05,
        lr_scheduler_type           = 'cosine',
        logging_steps               = 10,
        eval_strategy               = 'steps',
        eval_steps                  = 50,
        save_steps                  = 100,
        optim                       = 'adamw_8bit',
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        max_seq_length              = MAX_SEQ_LEN,
        report_to                   = sft_tracker,
        run_name                    = 'crisiscompute-sft',
        seed                        = 42,
    ),
    train_dataset   = sft_split['train'],
    eval_dataset    = sft_split['test'],
    processing_class = tokenizer,
    formatting_func = lambda x: x['text'],
)

print(f'\nStarting SFT training (tracking: {sft_tracker})...')
sft_result = sft_trainer.train()
print(f'\nSFT complete! Train loss: {sft_result.training_loss:.4f}')
sft_trainer.save_model('crisiscompute_sft/final')
tokenizer.save_pretrained('crisiscompute_sft/final')

In [ ]:
# GRPO Training
from trl import GRPOConfig, GRPOTrainer

def grpo_reward_fn(completions, **kwargs):
    scores = []
    for text in completions:
        text = text.strip()
        score = 0.0
        try:
            parsed = json.loads(text)
            score += 0.4
            action = parsed.get('action', '')
            if action in {'request_resources', 'wait', 'yield_to_urgent'}:
                score += 0.3
            if action in {'request_resources', 'yield_to_urgent'}:
                score += 0.2
            elif action == 'wait':
                score -= 0.3
            cores = int(parsed.get('cores_needed', 0))
            mem   = int(parsed.get('memory_needed', 0))
            gpu   = int(parsed.get('gpu_needed', 0))
            if 0 <= cores <= 16 and 0 <= mem <= 32 and gpu in (0, 1):
                score += 0.1
        except (json.JSONDecodeError, ValueError, TypeError):
            if '{' in text and '}' in text:
                score = 0.05
        scores.append(float(np.clip(score, -0.5, 1.0)))
    return scores

def format_grpo_prompt(sample):
    return tokenizer.apply_chat_template(sample['prompt'], tokenize=False,
                                          add_generation_prompt=True)

grpo_ds = Dataset.from_list(grpo_samples).shuffle(seed=42)
grpo_ds = grpo_ds.map(lambda x: {'prompt_text': format_grpo_prompt(x)})
grpo_ds = grpo_ds.rename_column('prompt_text', 'prompt_formatted')
grpo_ds = grpo_ds.select_columns(['prompt_formatted', 'reward'])
grpo_ds = grpo_ds.rename_column('prompt_formatted', 'prompt')

print(f'GRPO dataset: {len(grpo_ds)} samples')

grpo_tracker = 'wandb' if USE_WANDB else 'tensorboard'

grpo_trainer = GRPOTrainer(
    model            = model,
    args             = GRPOConfig(
        output_dir                  = 'crisiscompute_grpo',
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 1,
        learning_rate               = 5e-6,
        warmup_ratio                = 0.05,
        lr_scheduler_type           = 'cosine',
        logging_steps               = 10,
        save_steps                  = 100,
        optim                       = 'adamw_8bit',
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        max_prompt_length           = 512,
        max_completion_length       = 128,
        num_generations             = 2,
        report_to                   = grpo_tracker,
        run_name                    = 'crisiscompute-grpo',
        seed                        = 42,
    ),
    train_dataset    = grpo_ds,
    reward_funcs     = grpo_reward_fn,
    processing_class = tokenizer,
)

print(f'\nStarting GRPO training (tracking: {grpo_tracker})...')
grpo_result = grpo_trainer.train()
print('\nGRPO complete!')
grpo_trainer.save_model('crisiscompute_grpo/final')
tokenizer.save_pretrained('crisiscompute_grpo/final')

---

## Test Fine-Tuned Model

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user',   'content': (
        'Hour 3/8 | Scenario: gpu_outage_plus_urgent | Level: 3\n'
        'Available resources: GPU=0 (BUSY) | CPU=10/16 | RAM=20/32 GB\n'
        'Negotiation: fairness=0.81 | conflicts=1 | coalitions=1\n'
        'Your role: You train ML models. HIGH resource needs (2 CPU, 16 GB RAM, 1 GPU).\n'
        'What action do you take?'
    )},
]

inputs = tokenizer.apply_chat_template(
    test_prompt, tokenize=True, add_generation_prompt=True, return_tensors='pt'
).to(model.device)

with torch.no_grad():
    out = model.generate(input_ids=inputs, max_new_tokens=128,
                         temperature=0.3, do_sample=True,
                         pad_token_id=tokenizer.eos_token_id)

response = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
print('Model response:')
print(response)

try:
    parsed = json.loads(response.strip())
    print(f'\nValid JSON action: {parsed}')
except json.JSONDecodeError:
    print('\nNot valid JSON -- may need more training')

---

## Save & Push to Hugging Face

In [ ]:
model.save_pretrained_merged('crisiscompute_grpo/merged_16bit', tokenizer,
                              save_method='merged_16bit')
print('Saved merged 16-bit model')

# Uncomment to push to Hugging Face Hub:
# HF_USERNAME = 'Gautam0898'
# model.push_to_hub_merged(
#     f'{HF_USERNAME}/crisiscompute-grpo-lora',
#     tokenizer, save_method='lora',
#     token=os.environ.get('HF_TOKEN', ''),
# )

---

## Generated Artifacts Summary

In [ ]:
print('GENERATED ARTIFACTS')
print('='*70)

artifacts = [
    ('rl_before_after.png',       'RL: BEFORE vs AFTER training comparison'),
    ('rl_reward_curve.png',       'RL: 100-episode reward progression'),
    ('hybrid_before_after.png',   'Hybrid: BEFORE vs AFTER training comparison'),
    ('hybrid_reward_curve.png',   'Hybrid: 100-episode reward progression'),
    ('rl_vs_hybrid.png',          'Head-to-head: RL vs Hybrid bar chart'),
    ('rl_vs_hybrid_curves.png',   'Overlaid reward curves: RL vs Hybrid'),
    ('training_results.json',     'Full episode trajectories'),
    ('episode_metrics.json',      'Per-episode metrics'),
    ('selfplay_report.json',      'Theme #4: self-play results'),
    ('theme4_summary.json',       'Theme #4: curriculum + holdout summary'),
    ('baseline_comparison.json',  'Trained vs untrained baseline'),
    ('reward_curve.png',          'Auto-generated reward curve'),
    ('metrics_dashboard.png',     'Auto-generated metrics dashboard'),
]

for fname, desc in artifacts:
    fpath = Path('results') / fname
    if fpath.exists():
        size = fpath.stat().st_size / 1024
        print(f'  [OK] {fname:35s} ({size:6.1f} KB) -- {desc}')
    else:
        print(f'  [--] {fname:35s} (pending) -- {desc}')

print('\n' + '='*70)
print('Notebook complete!')